**Previously on:** all semester you've been handed a table and asked "what can you learn from
this?" Every table was *someone's decision* about which experiments to run. Lecture 23 found two
3D-printing features tied together perfectly (`bed_temperature`, `fan_speed`) and promised an
explanation later; Lecture 28 flagged that running many tests at once needs its own caution. Both
promises are cashed in today.

Today's topics:
* factorial designs, and why one-factor-at-a-time (OFAT) misses interactions
* randomization and replication -- how many replicates it takes to see an effect
* reverse-engineering the 3D-printing dataset's design, confounding and all
* the multiple-comparisons idea, at the level you need it

> By the end of today you should be able to complete Problems 1-3 of Homework 12: map and
> critique a provided factorial design, and design a small one of your own.

# From data consumer to data producer

Before anyone can analyze data, someone has to choose which conditions to measure. All semester
that someone was an anonymous printer operator or an API. Today it's you.

Two tiny experiment logs, no code yet. Log A: temperature 200, 220, 240 at fixed speed, then speed
40, 60, 80 at fixed temperature -- five runs, "efficient," never both at once. Log B: temperature
$\times$ speed crossed over two levels each -- four runs, every combination. Which one lets you say
anything about how temperature and speed *interact*? Hold that question; Section 2 answers it.

# Why one-factor-at-a-time misses interactions

Build a tiny, fully synthetic response surface where you, the author, know the ground truth --
something no real dataset can offer:

In [1]:
def true_strength(temp, speed):
    "Coded units: temp, speed in {-1, +1}. Unknown to the experimenter in real life."
    return 10 + 2 * temp + 3 * speed - 4 * temp * speed


print('OFAT arm 1 -- hold speed = -1, vary temp:')
for t in [-1, 0, 1]:
    print(f'  temp={t:+d}, speed=-1 -> strength={true_strength(t, -1):.1f}')

print('OFAT arm 2 -- hold temp = -1 (baseline), vary speed:')
for s in [-1, 0, 1]:
    print(f'  temp=-1, speed={s:+d} -> strength={true_strength(-1, s):.1f}')

OFAT arm 1 -- hold speed = -1, vary temp:
  temp=-1, speed=-1 -> strength=1.0
  temp=+0, speed=-1 -> strength=7.0
  temp=+1, speed=-1 -> strength=13.0
OFAT arm 2 -- hold temp = -1 (baseline), vary speed:
  temp=-1, speed=-1 -> strength=1.0
  temp=-1, speed=+0 -> strength=8.0
  temp=-1, speed=+1 -> strength=15.0


Raising temp alone (holding speed at -1) takes strength from 1.0 to 13.0. Raising speed alone
(holding temp at -1) takes it from 1.0 to 15.0. Both single-factor moves help a lot. An engineer
looking only at these two arms would reasonably guess: raise both, get more.

In [2]:
print('full 2x2 factorial -- all four corners:')
for t in [-1, 1]:
    for s in [-1, 1]:
        print(f'  temp={t:+d}, speed={s:+d} -> strength={true_strength(t, s):.1f}')

full 2x2 factorial -- all four corners:
  temp=-1, speed=-1 -> strength=1.0
  temp=-1, speed=+1 -> strength=15.0
  temp=+1, speed=-1 -> strength=13.0
  temp=+1, speed=+1 -> strength=11.0


The (+1, +1) corner -- raise both -- comes out at 11.0, *worse* than either single-factor change
(13.0 or 15.0). The `-4*temp*speed` term is an **interaction**: speed's effect depends on temp.
OFAT never sees this -- it never visits a corner where both factors leave baseline at once.
**OFAT can only detect main effects it probes one at a time -- it is structurally blind to
interactions.** In regression terms (Lecture 23): an interaction is just another design-matrix
column, `temp*speed` -- fittable only if the data actually varies both factors together.

# Vocabulary: factors, levels, factorials, and their cost

* **Factor** -- an input you control (temperature, print speed, infill density, ...).
* **Level** -- a specific setting of a factor (temp $\in$ {200, 220, 240}).
* **Full factorial design** -- every combination of every level of every factor. *k* factors at 2
  levels is a "$2^k$ design" -- the toy example was a $2^2$.
* **Interaction** -- one factor's effect depends on another's level. A full factorial is the
  minimum design that can *estimate* an interaction; OFAT cannot.

Cost: a full factorial grows as (levels)$^{\text{factors}}$ -- 3 factors at 3 levels is 27 runs,
5 at 3 levels is 243. Real experiments compromise: a **screening** (or **fractional**) design
deliberately runs fewer than every combination, trading some estimability (usually of interactions)
for far fewer runs when there are many factors. A named choice, not something stumbled into --
Section 5 checks whether a real dataset's compromise was a choice.

# Randomization and replication: how many runs does it take?

Two more ingredients, familiar under different names. **Replication** -- running a condition more
than once -- is Lecture 4-5's spread statistics applied to *runs*: without repeats you can't tell
a real change from "this measurement just landed here." **Randomization** -- random run order --
keeps drift (a warming print bed) from masquerading as a factor's effect, the fairness Lecture 28's
"is this difference real?" depends on.

Lecture 28 found `infill_pattern`'s 0.16 MPa gap was noise (p $\approx$ 0.95), and `material`'s
5.1 MPa gap real but borderline (p $\approx$ 0.041) -- was n=25 even enough? Simulate it instead of
trusting a formula: plant a known true difference `d` with noise matching the real data (std
$\approx$ 8.5 MPa), and count how often `ttest_ind` catches `d` at n replicates per group:

In [3]:
import numpy as np
from scipy import stats

rng = np.random.default_rng(219)
sigma = 8.5  # MPa, matches Lecture 28's group standard deviations
n_trials = 2000


def detection_rate(d, n, rng, n_trials=n_trials, sigma=sigma):
    "Fraction of simulated n-per-group experiments where ttest_ind catches a true gap of size d."
    hits = 0
    for _ in range(n_trials):
        g1 = rng.normal(0, sigma, n)
        g2 = rng.normal(d, sigma, n)
        if stats.ttest_ind(g1, g2).pvalue < 0.05:
            hits += 1
    return hits / n_trials


for n in [5, 10, 25, 50, 100]:
    print(f'n={n:3d} per group: detects a 3.0 MPa gap {detection_rate(3.0, n, rng):.0%} of the time')

n=  5 per group: detects a 3.0 MPa gap 7% of the time
n= 10 per group: detects a 3.0 MPa gap 11% of the time
n= 25 per group: detects a 3.0 MPa gap 24% of the time
n= 50 per group: detects a 3.0 MPa gap 43% of the time
n=100 per group: detects a 3.0 MPa gap 70% of the time


A real 3 MPa difference, run the way Lecture 28 ran `material` (n=25 per group), gets caught only
about a quarter of the time -- three out of four such experiments would come back "no evidence of a
difference," even though the difference is real. Plug in the two actual Lecture 28 gaps at n=25:

In [4]:
rng = np.random.default_rng(219)
print(f"n=25, gap=5.1 MPa (material):        detected {detection_rate(5.1, 25, rng):.0%} of the time")
print(f"n=25, gap=0.16 MPa (infill pattern): detected {detection_rate(0.16, 25, rng):.0%} of the time")

n=25, gap=5.1 MPa (material):        detected 53% of the time
n=25, gap=0.16 MPa (infill pattern): detected 6% of the time


The infill-pattern gap gets "detected" barely more often than pure chance (5%) -- consistent with a
gap that probably isn't real. The material gap gets caught only about half the time at n=25, which
is exactly why Lecture 28 called p $\approx$ 0.041 "borderline": at that sample size, a real 5 MPa
effect is close to a coin flip to detect. More replicates, not a fancier test, sharpens that call.

# Reveal: the 3D-printing dataset as a designed experiment

You've fit this table three times (Lectures 23, 24, 28) as "here are some numbers, model them."
Look at only the *input* columns today and ask a different question: what experiment produced these
50 rows?

## Dataset: 3D Printing (FDM) Parameters

Data come from [this github repository](https://github.com/ahmetokudan/3dprinterdeeplearning)
and are provided with an unlimited license (public domain).

In [5]:
import os
import pandas as pd

_file = '3dprinting.csv'
_github = f'https://raw.githubusercontent.com/wfreinhart/matse219/main/datasets/{_file}'
_local = next(
    (p for p in (f'datasets/{_file}', f'../datasets/{_file}', f'../../datasets/{_file}', f'../../../datasets/{_file}')
     if os.path.exists(p)),
    None,
)

try:
    data = pd.read_csv(_local if _local else _github)
except Exception as e:
    raise RuntimeError(
        f"Could not load '{_file}'. If you are in Colab, check your internet "
        f"connection and that the file exists at {_github}"
    ) from e
data.head()  # show a view of the data file

,layer_height (mm),wall_thickness (mm),infill_density (%),infill_pattern,nozzle_temperature (degC),bed_temperature (degC),print_velocity (mm/s),material,fan_speed (%),roughness (microns),tension_strength (Mpa),elongation (%)
0,0.02,8,90,grid,220,60,40,abs,0,25,18,1.2
1,0.02,7,90,honeycomb,225,65,40,abs,25,32,16,1.4
2,0.02,1,80,grid,230,70,40,abs,50,40,8,0.8
3,0.02,4,70,honeycomb,240,75,40,abs,75,68,10,0.5
4,0.02,6,90,grid,250,80,40,abs,100,92,5,0.7


In [6]:
factors = ['layer_height (mm)', 'wall_thickness (mm)', 'infill_density (%)', 'infill_pattern',
           'nozzle_temperature (degC)', 'bed_temperature (degC)', 'print_velocity (mm/s)',
           'material', 'fan_speed (%)']
data[factors].nunique()

layer_height (mm)             5
wall_thickness (mm)          10
infill_density (%)            9
infill_pattern                2
nozzle_temperature (degC)     9
bed_temperature (degC)        5
print_velocity (mm/s)         3
material                      2
fan_speed (%)                 5
dtype: int64

Nine candidate factors, most with 5-10 distinct levels. If this were even a modest full factorial
over those level counts:

In [7]:
import math

level_counts = data[factors].nunique()
print(f'theoretical full factorial size: {math.prod(level_counts):,} runs')
print(f'actual rows in this dataset:     {len(data)}')

theoretical full factorial size: 1,215,000 runs
actual rows in this dataset:     50


Over a million theoretical runs against 50 actual ones. Something other than "every combination of
every level" produced this table. Look one factor at a time first:

In [8]:
data['layer_height (mm)'].value_counts().sort_index()

layer_height (mm)
0.02    10
0.06    10
0.10    10
0.15    10
0.20    10
Name: count, dtype: int64

Five layer-height levels, 10 rows each -- looks like a clean single-factor sweep with replication.
Now look at two factors *together*:

In [9]:
data.groupby('layer_height (mm)')['print_velocity (mm/s)'].unique()

layer_height (mm)
0.02     [40]
0.06     [60]
0.10    [120]
0.15     [60]
0.20     [40]
Name: print_velocity (mm/s), dtype: object

Every row at a given `layer_height` has the *same* `print_velocity` -- 0.02 mm always pairs with 40
mm/s, 0.10 mm always with 120 mm/s, and so on. `layer_height` and `print_velocity` were never varied
independently: they're **perfectly confounded**. Whatever effect shows up on strength or roughness,
there is no way to attribute it to layer height, to print velocity, or to their interaction -- the
data cannot distinguish the three.

# Critiquing the design

Push further -- are `material`, `nozzle_temperature`, `bed_temperature`, and `fan_speed` independent
of each other, or bundled?

In [10]:
data[['material', 'nozzle_temperature (degC)', 'bed_temperature (degC)',
      'fan_speed (%)']].drop_duplicates()

,material,nozzle_temperature (degC),bed_temperature (degC),fan_speed (%)
0,abs,220,60,0
1,abs,225,65,25
2,abs,230,70,50
3,abs,240,75,75
4,abs,250,80,100
5,pla,200,60,0
6,pla,205,65,25
7,pla,210,70,50
8,pla,215,75,75
9,pla,220,80,100


Ten rows, five per material: nozzle/bed temperature and fan speed step together as a fixed "recipe"
(ABS runs hot, PLA cooler). Reasonable practice -- you would not print PLA at 250 degC -- but it
entangles `material`'s effect with three other factors at once. Check exactly how entangled, the
way Lecture 23 flagged but didn't run:

In [11]:
print("corr(bed_temperature, fan_speed):     ", data['bed_temperature (degC)'].corr(data['fan_speed (%)']))
print("corr(nozzle_temperature, bed_temperature):", data['nozzle_temperature (degC)'].corr(data['bed_temperature (degC)']))

corr(bed_temperature, fan_speed):      1.0
corr(nozzle_temperature, bed_temperature): 0.6024533658898499


`bed_temperature` and `fan_speed` correlate at exactly 1.0 -- a perfectly rigid lockstep, not just
"tends to move together." No fitting procedure can split credit between two columns carrying
identical information; `lstsq` in Lecture 23 wasn't confused, the *design* left nothing to separate.
Contrast with `wall_thickness`/`infill_density`, which look independently varied:

In [12]:
data.groupby('layer_height (mm)')['infill_density (%)'].apply(lambda s: (s.min(), s.max()))

layer_height (mm)
0.02    (10, 90)
0.06    (10, 90)
0.10    (30, 90)
0.15    (10, 80)
0.20    (20, 90)
Name: infill_density (%), dtype: object

Infill density spans roughly its full 10-90% range inside every layer-height group -- genuinely
independent, unlike `print_velocity`. Design critique:
* **Varied independently enough:** `wall_thickness`, `infill_density`, `infill_pattern`.
* **Confounded by construction:** `layer_height` $\leftrightarrow$ `print_velocity` (perfectly);
  `material` $\leftrightarrow$ {`nozzle_temperature`, `bed_temperature`, `fan_speed`} (bundled into
  recipes, `bed_temperature`/`fan_speed` exactly so).
* **Replication:** 10 rows per `layer_height` looks like replication, but `print_velocity` rides
  along -- not true replicates.
* **Randomization:** no run-order column exists -- we can't check for drift.

A real, published dataset used for regression three times this semester still has a design flaw
limiting its causal claims. **Guardrail:** the Lecture 23 model is still a valid *predictive* tool
inside the recipes tested -- this is about causal separability, not predictive accuracy.

# An honest paragraph on running many tests

A grid like this tempts you to test *everything* -- every factor against every outcome -- and
report whichever pair comes back significant. Lecture 28 flagged the risk without running it; run
it now. Simulate 20 truly-null comparisons, many times:

In [13]:
rng = np.random.default_rng(219)
n_trials, n_tests = 5000, 20
false_positives = []
for _ in range(n_trials):
    hits = sum(stats.ttest_ind(rng.normal(0, 1, 20), rng.normal(0, 1, 20)).pvalue < 0.05
               for _ in range(n_tests))
    false_positives.append(hits)

print(f'average "significant" (p<0.05) results per family of {n_tests} truly-null tests: '
      f'{np.mean(false_positives):.2f}')
print(f'fraction of families with at least one false positive: {np.mean(np.array(false_positives) >= 1):.0%}')

average "significant" (p<0.05) results per family of 20 truly-null tests: 1.00
fraction of families with at least one false positive: 64%


About one false "significant" result per 20 null tests, on average, with a 64% chance of at least
one per family -- test enough pairs on this dataset's 9 factors and 3 outcomes and something will
look significant by chance alone. Fix: **Bonferroni correction** -- for $m$ comparisons, compare
each p-value against $\alpha / m$ instead of $\alpha$, keeping the *family's* false-positive rate
near 5%. Conservative, but better than reporting one flashy p-value and staying quiet about the rest.

# Designing it better, and the HW12 bridge

One more connection to Lecture 23's `lstsq` fit, in one number: a design's confounding shows up
directly as an ill-conditioned design matrix.

In [14]:
X_confounded = np.column_stack([np.ones(len(data)), data['bed_temperature (degC)'], data['fan_speed (%)']])
print('condition number, [1, bed_temperature, fan_speed]:', np.linalg.cond(X_confounded))

condition number, [1, bed_temperature, fan_speed]: 5.92852542026332e+17


That large a condition number is numerically singular -- `lstsq` returns *an* answer, not a
trustworthy one. A well-designed experiment keeps that number small.

A better design: cross `layer_height` and `print_velocity` independently at 3 levels (9 runs) with
2-3 replicates (18-27 runs) and a recorded run order -- smaller than 50 rows, far more informative.

Homework 12: map and critique a provided factorial design, analyze provided results (main effects,
one interaction, a t-test with today's multiple-comparisons caveat), then design a small factorial
of your own. Lectures 32-37 taught you to consume someone else's designed data responsibly; today
closes the loop by teaching you to design your own.

## [Check your understanding]

Compute `corr(nozzle_temperature, bed_temperature)`. It's high but *not* exactly 1.0 like
`bed_temperature`/`fan_speed` (`nozzle_temperature` steps unevenly for ABS: 5, 5, 10, 10 degC).
In 1-2 sentences: why is this pair only *partially* confounded, and does that partial confounding
still limit what you can conclude about `nozzle_temperature` alone?

# Wrap-up

* A full factorial is the minimum design that can estimate an interaction; OFAT cannot.
* Replication and randomization let you tell a real effect from noise -- needed replicates scale
  with the effect size you're chasing.
* 3D-printing is a real, *confounded* design: `layer_height`/`print_velocity` and
  `material`/`nozzle_temperature`/`bed_temperature`/`fan_speed` were never varied independently.
* Many tests on one grid inflates false positives -- Bonferroni ($\alpha / m$) is the fix.

**Take-home message:** every dataset was produced by someone's design choices, and a model that
fits well can still be powerless to answer a question the design never set up to answer.

Next class: Quiz 8, a fresh worked DOE example, and a whole-course synthesis pass.